In [52]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib
DATA_PATH = Path("../data/raw/student_mental_health.csv")
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
display(df.head())
display(df.dtypes.to_frame("dtype"))
display(df.isna().sum().loc[lambda s: s.gt(0)].to_frame("missing"))
print("Exact duplicate rows:", df.duplicated().sum())

Shape: (2000, 17)


,Student_ID,Age,Gender,Year_of_Study,Sleep_Hours,Study_Hours_Per_Week,GPA,Social_Media_Hours,Screen_Time_Hours,Physical_Activity,Financial_Stress,Family_Support,Attendance_Rate,Part_Time_Job,Diet_Quality,Previous_Mental_Health_History,Mental_Health_Risk
0,STU0001,22,Male,3,6.6,17.6,2.91,6.9,7.5,Moderate,8.5,4.0,78.0,No,Poor,No,High
1,STU0002,21,Male,4,5.6,22.4,2.92,7.6,11.5,Low,4.5,5.3,85.4,Yes,Average,No,Medium
2,STU0003,22,Male,5,7.1,28.1,2.68,6.5,8.9,Low,4.8,6.1,79.5,No,Average,No,Medium
3,STU0004,24,Male,1,4.5,14.8,2.23,5.5,7.6,High,3.7,8.0,73.9,No,Average,Yes,High
4,STU0005,20,Female,1,6.7,17.2,2.69,4.3,7.0,Moderate,8.0,7.4,99.8,Yes,NaN,No,Medium


,dtype
Student_ID,str
Age,int64
Gender,str
Year_of_Study,int64
Sleep_Hours,float64
Study_Hours_Per_Week,float64
GPA,float64
Social_Media_Hours,float64
Screen_Time_Hours,float64
Physical_Activity,str


,missing
Sleep_Hours,62
GPA,38
Social_Media_Hours,58
Financial_Stress,91
Family_Support,54
Attendance_Rate,47
Diet_Quality,69


Exact duplicate rows: 0


In [53]:
TARGET = "Mental_Health_Risk"
display(df[TARGET].value_counts().to_frame("count"))
display((df[TARGET].value_counts(normalize=True) * 100).round(2).to_frame("percent"))

,count
Mental_Health_Risk,
High,680
Medium,660
Low,660


,percent
Mental_Health_Risk,
High,34.0
Medium,33.0
Low,33.0


In [54]:
numeric_all = df.select_dtypes(include="number").columns
outliers_rows = []

for col in numeric_all:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    count = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()
    outliers_rows.append({
        "feature": col, "iqr_lower": lower_bound, "iqr_upper": upper_bound,
        "potential_outliers": int(count)
    })
display(pd.DataFrame(outliers_rows))

,feature,iqr_lower,iqr_upper,potential_outliers
0,Age,15.5000,27.5000,4
1,Year_of_Study,-3.5000,8.5000,0
2,Sleep_Hours,3.6000,10.0000,2
3,Study_Hours_Per_Week,4.0000,36.0000,7
4,GPA,1.6150,3.4950,9
5,Social_Media_Hours,3.0125,8.9125,20
6,Screen_Time_Hours,3.9500,13.1500,7
7,Financial_Stress,0.1500,11.7500,0
8,Family_Support,0.7500,12.3500,0
9,Attendance_Rate,65.7500,104.5500,12


In [55]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder


class IQRCapper(BaseEstimator, TransformerMixin):
    """Cap numerical values at the training-set IQR bounds."""

    def __init__(self, factor=1.5):
        self.factor = factor

    def fit(self, X, y=None):
        X = X.copy()
        self.feature_names_in_ = X.columns
        q1 = X.quantile(0.25)
        q3 = X.quantile(0.75)
        iqr = q3 - q1
        self.lower_bounds_ = q1 - self.factor * iqr
        self.upper_bounds_ = q3 + self.factor * iqr
        return self

    def transform(self, X):
        X = X.copy()
        return X.clip(
            lower=self.lower_bounds_,
            upper=self.upper_bounds_,
            axis="columns",
        )


ID_COLUMN = "Student_ID"
OUTPUT_DIR = Path("../data/processed/")
X = df.drop(columns=[TARGET, ID_COLUMN], errors="ignore")
y_text = df[TARGET].astype(str)

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

X_train, X_test, y_train_text, y_test_text = train_test_split(
    X,
    y_text,
    test_size=0.2,
    random_state=42,
    stratify=y_text
)
X_train.shape, y_train_text.shape

((1600, 15), (1600,))

In [56]:
numeric_pipe = Pipeline(
    steps=[
        ("iqr_capper", IQRCapper(factor=1.5)),
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipe, numeric_features),
        ("categorical", categorical_pipe, categorical_features),
    ],
    remainder="drop",
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)
X_train_processed


array([[-0.0637065 , -0.47163633, -1.40418192, ...,  0.        ,
         1.        ,  0.        ],
       [ 0.87574418, -1.2323401 , -1.6760358 , ...,  0.        ,
         1.        ,  0.        ],
       [-1.00315718, -0.47163633,  0.40817727, ...,  0.        ,
         1.        ,  0.        ],
       ...,
       [-1.94260785,  1.81047496,  0.04570543, ...,  0.        ,
         1.        ,  0.        ],
       [ 0.40601884,  1.04977119, -0.7698562 , ...,  0.        ,
         1.        ,  0.        ],
       [-0.53343184, -1.2323401 , -0.95109212, ...,  0.        ,
         0.        ,  1.        ]], shape=(1600, 23))

In [57]:
target_encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1,
)
y_train = pd.Series(
    target_encoder.fit_transform(y_train_text.to_numpy().reshape(-1, 1)).ravel(),
    index=y_train_text.index,
    name=TARGET
)

y_test = pd.Series(
    target_encoder.transform(y_test_text.to_numpy().reshape(-1, 1)).ravel(),
    index=y_test_text.index,
    name=TARGET,
)
display(y_train)


726     2
1384    2
1579    2
1098    1
582     2
       ..
1885    2
800     1
1163    2
891     1
1297    0
Name: Mental_Health_Risk, Length: 1600, dtype: int64

In [59]:
feature_names = preprocessor.get_feature_names_out()

X_train_processed = pd.DataFrame(
    X_train_processed, columns=feature_names, index=X_train.index
)
X_test_processed = pd.DataFrame(
    X_test_processed, columns=feature_names, index=X_test.index
)
display(X_train_processed)

,numeric__Age,numeric__Year_of_Study,numeric__Sleep_Hours,numeric__Study_Hours_Per_Week,numeric__GPA,numeric__Social_Media_Hours,numeric__Screen_Time_Hours,numeric__Financial_Stress,numeric__Family_Support,numeric__Attendance_Rate,...,categorical__Physical_Activity_High,categorical__Physical_Activity_Low,categorical__Physical_Activity_Moderate,categorical__Part_Time_Job_No,categorical__Part_Time_Job_Yes,categorical__Diet_Quality_Average,categorical__Diet_Quality_Good,categorical__Diet_Quality_Poor,categorical__Previous_Mental_Health_History_No,categorical__Previous_Mental_Health_History_Yes
726,-0.063706,-0.471636,-1.404182,-0.238603,0.074214,0.148597,1.098992,0.713748,0.643008,-1.244652,...,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
1384,0.875744,-1.232340,-1.676036,-1.330997,-0.976093,0.327697,0.194658,-0.842691,0.368687,-0.465696,...,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
1579,-1.003157,-0.471636,0.408177,-0.516037,-1.034443,-0.299153,-0.830254,-0.305988,0.752736,0.772288,...,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
1098,0.875744,-1.232340,1.404975,-0.082547,0.891120,0.775447,1.098992,0.821088,-0.948052,-0.215317,...,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
582,0.406019,0.289067,0.408177,0.663055,-0.275888,-0.388703,-0.347943,0.660077,0.697872,-1.453301,...,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1885,0.406019,0.289067,-1.494800,-1.088243,0.074214,-0.299153,-0.830254,0.177045,1.081921,1.690344,...,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0
800,0.875744,-0.471636,-0.407384,-0.273283,0.365966,-0.657353,0.676970,-0.628009,1.081921,-0.507426,...,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0
1163,-1.942608,1.810475,0.045705,0.298923,0.716069,0.596347,0.013791,1.035769,-0.015362,0.146341,...,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0
891,0.406019,1.049771,-0.769856,0.749752,0.599368,-0.836452,-1.312566,-0.950031,0.149230,0.007242,...,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
X_train_processed.to_csv(OUTPUT_DIR / "X_train_processed.csv", index=False)
X_test_processed.to_csv(OUTPUT_DIR / "X_test_processed.csv", index=False)
y_train.to_csv(OUTPUT_DIR / "y_train.csv", index=False)
y_test.to_csv(OUTPUT_DIR / "y_test.csv", index=False)
joblib.dump(preprocessor, OUTPUT_DIR / "preprocessor.joblib")
joblib.dump(target_encoder, OUTPUT_DIR / "target_ordinal_encoder.joblib")

print("Preprocessing completed.")
print(f"Original rows after duplicate removal: {len(df)}")
print(f"Train rows: {len(X_train_processed)} | Test rows: {len(X_test_processed)}")
print(f"Numeric features ({len(numeric_features)}): {numeric_features}")
print(f"Categorical features ({len(categorical_features)}): {categorical_features}")
print("Training target distribution:")
print(y_train_text.value_counts(normalize=True).round(3))
print(f"Saved outputs to: {OUTPUT_DIR.resolve()}")

Preprocessing completed.
Original rows after duplicate removal: 2000
Train rows: 1600 | Test rows: 400
Numeric features (10): ['Age', 'Year_of_Study', 'Sleep_Hours', 'Study_Hours_Per_Week', 'GPA', 'Social_Media_Hours', 'Screen_Time_Hours', 'Financial_Stress', 'Family_Support', 'Attendance_Rate']
Categorical features (5): ['Gender', 'Physical_Activity', 'Part_Time_Job', 'Diet_Quality', 'Previous_Mental_Health_History']
Training target distribution:
Mental_Health_Risk
High      0.34
Medium    0.33
Low       0.33
Name: proportion, dtype: float64
Saved outputs to: /home/alaric/Desktop/studies/projects/ML/mental-health-risk-prediction/data/processed
